# Hybrid Legal Retrieval with Qwen3-Embedding & Qwen3-Reranker

**Improvements over baseline:**
1. **Chunking + MaxSim reranking** — split long docs into overlapping chunks, score each, take max per doc
2. **Multi-query court retrieval** — original + expanded + law abbreviation queries
3. **Adaptive citation count** — return more when reranker is confident
4. **FAISS caching** — saves index to disk so restarts are instant

In [1]:
!pip install -q sentence_transformers bm25s faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.5/74.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 71.9 MB/s eta 0:00:00:00:0100:01


## Configuration

In [2]:
import pandas as pd
import numpy as np
import time
import warnings
import gc
import os
import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer
import bm25s
import faiss
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

# --- Paths ---
TRAIN_PATH = '/kaggle/input/competitions/llm-agentic-legal-information-retrieval/train.csv'
VAL_PATH = '/kaggle/input/competitions/llm-agentic-legal-information-retrieval/val.csv'
TEST_PATH = '/kaggle/input/competitions/llm-agentic-legal-information-retrieval/test.csv'
LAWS_PATH = '/kaggle/input/competitions/llm-agentic-legal-information-retrieval/laws_de.csv'
COURT_PATH = '/kaggle/input/competitions/llm-agentic-legal-information-retrieval/court_considerations.csv'
OUTPUT_PATH = 'submission.csv'
FAISS_CACHE = '/kaggle/working/faiss_laws.index'

# --- Models ---
EMBEDDING_MODEL_NAME = 'Qwen/Qwen3-Embedding-0.6B'
RERANKER_MODEL_NAME = 'Qwen/Qwen3-Reranker-0.6B'

# --- Task instruction ---
TASK_INSTRUCTION = (
    'Given a legal question in English, retrieve relevant Swiss law articles '
    'and court decisions written in German.'
)

# --- Hyperparameters ---
TOP_K_RETRIEVAL = 60       # candidates from BM25/FAISS before reranking
TOP_K_FINAL = 10           # final citations after reranking (baseline default)
TEXT_TRUNCATE = 3500        # max chars for reranker input
RRF_K = 60                 # reciprocal rank fusion parameter
LAWS_EXPANSION_TOP_K = 3   # law citations used for court query expansion
EMBEDDING_BATCH_SIZE = 16
RERANKER_BATCH_SIZE = 8

# --- Chunking params (NEW) ---
CHUNK_SIZE = 200            # words per chunk
CHUNK_OVERLAP = 50          # overlapping words between chunks

# --- GPU allocation ---
USE_GPU = torch.cuda.is_available()
NUM_GPUS = torch.cuda.device_count() if USE_GPU else 0

if NUM_GPUS >= 2:
    DEVICE_EMB = 'cuda:0'
    DEVICE_RERANK = 'cuda:1'
elif NUM_GPUS == 1:
    DEVICE_EMB = 'cuda:0'
    DEVICE_RERANK = 'cuda:0'
else:
    DEVICE_EMB = 'cpu'
    DEVICE_RERANK = 'cpu'

print(f'GPUs: {NUM_GPUS} | Embedding: {DEVICE_EMB} | Reranker: {DEVICE_RERANK}')

GPUs: 2 | Embedding: cuda:0 | Reranker: cuda:1


## Data Loading

In [3]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

laws_df = pd.read_csv(LAWS_PATH)
laws_df['text'] = laws_df['text'].fillna('')
laws_df['title'] = laws_df['title'].fillna('') if 'title' in laws_df.columns else ''

court_df = pd.read_csv(COURT_PATH, usecols=['citation', 'text'])
court_df['text'] = court_df['text'].fillna('')

print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print(f'Laws: {len(laws_df)} | Court: {len(court_df)}')

Train: 1139 | Val: 10 | Test: 40
Laws: 175933 | Court: 2476315


## Text Composition

In [4]:
# Laws: citation + title + text (richer representation)
laws_texts = (
    laws_df['citation'] + ' ' + laws_df['title'] + ' ' + laws_df['text']
).tolist()

# Court: citation + text
court_texts = (
    court_df['citation'] + ' ' + court_df['text']
).tolist()

# Lookup dicts for reranking
law_citation_to_text = dict(zip(laws_df['citation'], laws_texts))
court_citation_to_text = dict(zip(court_df['citation'], court_texts))

print(f'Laws text sample: {laws_texts[0][:120]}...')
print(f'Court text sample: {court_texts[0][:120]}...')

Laws text sample: Art. 1 112 Übereinkunft vom 22. Juni 1875 zwischen dem Schweizerischen Bundesrate und dem Einwohnergemeinderate der Stad...
Court text sample: BGE 139 I 2 E. 1.12.2011 betr. Verweigerung der Beiladung seien gutzuheissen. BGE 139 I 2 S. 7...


## BM25 Lexical Index

In [5]:
import os

BM25_LAWS_CACHE = '/kaggle/working/bm25_laws'
BM25_COURTS_CACHE = '/kaggle/working/bm25_courts'

def build_bm25_index(texts, name=''):
    truncated = [str(t).lower()[:TEXT_TRUNCATE] for t in texts]
    tokens = bm25s.tokenize(truncated, stopwords='de')
    retriever = bm25s.BM25()
    retriever.index(tokens)
    del tokens, truncated
    gc.collect()
    print(f'BM25 index ready: {name}')
    return retriever

def search_bm25(retriever, query, top_k):
    query_tokens = bm25s.tokenize([query.lower()], stopwords='de')
    docs, scores = retriever.retrieve(query_tokens, k=top_k)
    return [(int(docs[0][i]), float(scores[0][i])) for i in range(len(docs[0]))]

# Laws BM25
if os.path.exists(BM25_LAWS_CACHE):
    bm25_laws = bm25s.BM25.load(BM25_LAWS_CACHE)
    print('BM25 laws loaded from cache')
else:
    bm25_laws = build_bm25_index(laws_texts, 'laws')
    bm25_laws.save(BM25_LAWS_CACHE)
    print('BM25 laws saved to cache')

# Courts BM25
if os.path.exists(BM25_COURTS_CACHE):
    bm25_courts = bm25s.BM25.load(BM25_COURTS_CACHE)
    print('BM25 courts loaded from cache')
else:
    bm25_courts = build_bm25_index(court_texts, 'courts')
    bm25_courts.save(BM25_COURTS_CACHE)
    print('BM25 courts saved to cache')

Split strings:   0%|          | 0/175933 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/175933 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/175933 [00:00<?, ?it/s]

BM25 index ready: laws
BM25 laws saved to cache


Split strings:   0%|          | 0/2476315 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/2476315 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/2476315 [00:00<?, ?it/s]

BM25 index ready: courts
BM25 courts saved to cache


## Qwen3-Embedding

In [6]:
print(f'Loading embedding model: {EMBEDDING_MODEL_NAME} on {DEVICE_EMB}')
embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME,
    device=DEVICE_EMB,
    model_kwargs={'torch_dtype': torch.float16},
)
embedding_model.max_seq_length = 512
print('Embedding model ready')

Loading embedding model: Qwen/Qwen3-Embedding-0.6B on cuda:0


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Embedding model ready


In [7]:
def encode_queries(model, queries):
    "Encode queries WITH instruction prefix (Qwen3 convention)."
    prefixed = [
        f'Instruct: {TASK_INSTRUCTION}\nQuery: {q}'
        for q in queries
    ]
    return model.encode(
        prefixed,
        batch_size=32,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True,
    )


def encode_corpus(model, texts):
    "Encode documents WITHOUT prefix (Qwen3 convention)."
    truncated = [str(t)[:TEXT_TRUNCATE] for t in texts]
    return model.encode(
        truncated,
        batch_size=EMBEDDING_BATCH_SIZE,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True,
    )

## Laws Corpus Encoding + FAISS Index (with disk cache)

In [8]:
if os.path.exists(FAISS_CACHE):
    faiss_index = faiss.read_index(FAISS_CACHE)
    print(f'FAISS loaded from cache: {faiss_index.ntotal} vectors')
else:
    print('Encoding laws corpus...')
    start = time.time()
    laws_embeddings = encode_corpus(embedding_model, laws_texts)
    laws_embeddings = np.array(laws_embeddings, dtype=np.float32)
    print(f'Laws encoding done: {laws_embeddings.shape} in {time.time()-start:.1f}s')

    faiss_index = faiss.IndexFlatIP(laws_embeddings.shape[1])
    faiss_index.add(laws_embeddings)

    faiss.write_index(faiss_index, FAISS_CACHE)
    print(f'FAISS index built and saved: {faiss_index.ntotal} vectors')

    del laws_embeddings
    gc.collect()
    torch.cuda.empty_cache()

Encoding laws corpus...


Batches:   0%|          | 0/10996 [00:00<?, ?it/s]

Laws encoding done: (175933, 1024) in 2593.5s
FAISS index built and saved: 175933 vectors


## Qwen3-Reranker with Chunking + MaxSim

**Key improvement**: Instead of truncating long documents, split them into overlapping chunks,
score each chunk independently, and assign the **max chunk score** to the parent document.
This finds relevant passages buried deep in long court decisions.

In [9]:
class Qwen3Reranker:
    "Qwen3-Reranker-0.6B using AutoModelForCausalLM with yes/no probability."

    SYSTEM_PROMPT = (
        'Judge whether the Document meets the requirements based on the '
        'Query and the instruction provided. Note that the answer can only be '
        '"yes" or "no".'
    )

    def __init__(self, model_name, device='cuda', dtype=torch.float16):
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name, padding_side='left', trust_remote_code=True,
        )
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype=dtype, trust_remote_code=True,
        ).to(device).eval()
        self.device = device
        self.yes_id = self.tokenizer.convert_tokens_to_ids('yes')
        self.no_id = self.tokenizer.convert_tokens_to_ids('no')

    def _format_pair(self, instruction, query, document):
        return (
            f'<|im_start|>system\n{self.SYSTEM_PROMPT}<|im_end|>\n'
            f'<|im_start|>user\n'
            f'<Instruct>: {instruction}\n'
            f'<Query>: {query}\n'
            f'<Document>: {document}\n'
            f'<|im_end|>\n'
            f'<|im_start|>assistant\n<think>\n\n</think>\n\n'
        )

    @torch.no_grad()
    def predict(self, pairs, batch_size=8, instruction=None):
        if instruction is None:
            instruction = TASK_INSTRUCTION
        all_scores = []
        for start in range(0, len(pairs), batch_size):
            batch = pairs[start:start + batch_size]
            prompts = [
                self._format_pair(instruction, q, d[:TEXT_TRUNCATE])
                for q, d in batch
            ]
            inputs = self.tokenizer(
                prompts, padding=True, truncation=True,
                max_length=4096, return_tensors='pt',
            ).to(self.device)
            logits = self.model(**inputs).logits[:, -1, :]
            yes_no = torch.stack(
                [logits[:, self.no_id], logits[:, self.yes_id]], dim=1
            )
            probs = F.softmax(yes_no, dim=1)
            scores = probs[:, 1].cpu().numpy()
            all_scores.extend(scores.tolist())
        return all_scores


print(f'Loading reranker: {RERANKER_MODEL_NAME} on {DEVICE_RERANK}')
reranker = Qwen3Reranker(RERANKER_MODEL_NAME, device=DEVICE_RERANK)
print('Reranker ready')

Loading reranker: Qwen/Qwen3-Reranker-0.6B on cuda:1


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

Reranker ready


## Retrieval Functions

In [10]:
def search_faiss(index, query_embedding, top_k):
    "Dense retrieval using FAISS inner-product index."
    D, I = index.search(query_embedding.reshape(1, -1), top_k)
    return [(int(I[0][i]), float(D[0][i])) for i in range(len(I[0]))]


def reciprocal_rank_fusion(sparse_results, dense_results, k=RRF_K):
    "Merge BM25 and dense rankings using weighted RRF."
    scores = {}
    for rank, (idx, _) in enumerate(sparse_results):
        scores[idx] = scores.get(idx, 0) + 0.5 / (k + rank)   # BM25 weight down
    for rank, (idx, _) in enumerate(dense_results):
        scores[idx] = scores.get(idx, 0) + 1.5 / (k + rank)   # FAISS weight up
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [idx for idx, _ in ranked]


def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    "Split text into overlapping word chunks."
    words = str(text).split()
    if len(words) <= chunk_size:
        return [str(text)]
    chunks = []
    for i in range(0, len(words), chunk_size - overlap):
        chunk = ' '.join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks


def rerank_candidates(query, candidates):
    "Rerank with MaxSim: chunk docs, score each chunk, take max per doc."
    if not candidates:
        return []

    # Build all (query, chunk) pairs and track parent mapping
    chunk_pairs = []
    chunk_mapping = []  # index into candidates list
    for c_idx, (citation, text) in enumerate(candidates):
        chunks = chunk_text(text)
        for chunk in chunks:
            chunk_pairs.append([query, chunk])
            chunk_mapping.append(c_idx)

    # Score all chunks
    chunk_scores = reranker.predict(chunk_pairs, batch_size=RERANKER_BATCH_SIZE)

    # MaxSim: take max chunk score per document
    doc_max_scores = {}
    for score, parent_idx in zip(chunk_scores, chunk_mapping):
        citation = candidates[parent_idx][0]
        if citation not in doc_max_scores or score > doc_max_scores[citation]:
            doc_max_scores[citation] = score

    # Sort by score descending
    scored = sorted(doc_max_scores.items(), key=lambda x: x[1], reverse=True)
    return scored

## Main Retrieval Pipeline

Improvements over baseline:
1. **Multi-query court retrieval** — original + expanded + law abbreviation queries
2. **Chunking + MaxSim reranking** — finds relevant passages in long documents
3. **Adaptive citation count** — returns more when reranker is confident

In [12]:
def retrieve_citations(query, query_embedding):
    "Full retrieval pipeline for a single query."

    # 1. Laws: hybrid search (BM25 + FAISS + RRF)
    sparse_results = search_bm25(bm25_laws, query, TOP_K_RETRIEVAL)
    dense_results = search_faiss(faiss_index, query_embedding, TOP_K_RETRIEVAL)
    fused_indices = reciprocal_rank_fusion(sparse_results, dense_results)
    law_citations = [
        laws_df.iloc[idx]['citation']
        for idx in fused_indices[:TOP_K_RETRIEVAL]
        if 0 <= idx < len(laws_df)
    ]

    # 2. Court: multi-query BM25 for better recall
    court_idx_scores = {}

    # Query A: original English query
    for idx, score in search_bm25(bm25_courts, query, TOP_K_RETRIEVAL):
        if 0 <= idx < len(court_df):
            if idx not in court_idx_scores or score > court_idx_scores[idx]:
                court_idx_scores[idx] = score

    # Query B: expanded with top law citations
    expansion = ' '.join(law_citations[:LAWS_EXPANSION_TOP_K])
    if expansion:
        expanded_query = f'{query} {expansion}'
        for idx, score in search_bm25(bm25_courts, expanded_query, TOP_K_RETRIEVAL):
            if 0 <= idx < len(court_df):
                if idx not in court_idx_scores or score > court_idx_scores[idx]:
                    court_idx_scores[idx] = score

    # Query C: law abbreviations only (StPO, OR, ZGB, BV etc.)
    abbrevs = set()
    for cit in law_citations[:10]:
        for part in cit.split():
            if part.isupper() and len(part) >= 2:
                abbrevs.add(part)
    if abbrevs:
        kw_query = ' '.join(abbrevs)
        for idx, score in search_bm25(bm25_courts, kw_query, TOP_K_RETRIEVAL):
            if 0 <= idx < len(court_df):
                if idx not in court_idx_scores or score > court_idx_scores[idx]:
                    court_idx_scores[idx] = score

    court_sorted = sorted(court_idx_scores.items(), key=lambda x: x[1], reverse=True)
    court_citations = [
        court_df.iloc[idx]['citation']
        for idx, _ in court_sorted[:TOP_K_RETRIEVAL]
        if 0 <= idx < len(court_df)
    ]

    # 3. Merge & deduplicate
    seen = set()
    all_citations = []
    for cit in law_citations + court_citations:
        if cit not in seen:
            all_citations.append(cit)
            seen.add(cit)

    # 4. Prepare candidates for reranking
    candidates = []
    for cit in all_citations[:TOP_K_RETRIEVAL]:
        text = law_citation_to_text.get(cit) or court_citation_to_text.get(cit, '')
        if text:
            candidates.append((cit, text))

    # 5. Rerank with MaxSim chunking
    reranked = rerank_candidates(query, candidates)

    # 6. Adaptive citation count
    # Return all citations above threshold, minimum 5, maximum 30
    result = []
    for cit, score in reranked:
        if score >= 0.5 or len(result) < TOP_K_FINAL:
            result.append(cit)
    return result[:30]

## Evaluation on Validation Set

In [13]:
def compute_macro_f1(all_predicted, all_gold):
    "Citation-level Macro F1 (competition metric)."
    f1_scores = []
    for pred, gold in zip(all_predicted, all_gold):
        pred_set = set(pred)
        gold_set = set(gold)
        if not gold_set:
            continue
        tp = len(pred_set & gold_set)
        precision = tp / len(pred_set) if pred_set else 0.0
        recall = tp / len(gold_set) if gold_set else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        f1_scores.append(f1)
    return np.mean(f1_scores) if f1_scores else 0.0


# Encode validation queries
val_queries = val_df['query'].tolist()
print(f'Encoding {len(val_queries)} validation queries...')
val_query_embeddings = encode_queries(embedding_model, val_queries)

# Run retrieval on validation set
print('Running validation...')
val_predictions = []
val_gold = []

for i, row in val_df.iterrows():
    pred = retrieve_citations(row['query'], val_query_embeddings[i])
    val_predictions.append(pred)
    gold = [c.strip() for c in str(row.get('gold_citations', '')).split(';') if c.strip()]
    val_gold.append(gold)

val_f1 = compute_macro_f1(val_predictions, val_gold)
print(f'\nValidation Macro F1: {val_f1:.5f}')
print(f'Baseline:            0.06606\n')

# Per-query breakdown
for i, (pred, gold) in enumerate(zip(val_predictions, val_gold)):
    pred_set, gold_set = set(pred), set(gold)
    tp = len(pred_set & gold_set)
    p = tp / len(pred_set) if pred_set else 0
    r = tp / len(gold_set) if gold_set else 0
    f = 2*p*r/(p+r) if (p+r) > 0 else 0
    print(f'  Query {i+1}: F1={f:.4f} | P={p:.4f} R={r:.4f} | pred={len(pred)} gold={len(gold)} tp={tp}')

# Debug: check retrieval recall
print('\n--- Retrieval Recall Debug ---')
for i, (pred, gold) in enumerate(zip(val_predictions, val_gold)):
    gold_set = set(gold)
    found = sum(1 for c in pred if c in gold_set)
    print(f'  Q{i+1}: found {found}/{len(gold)} gold citations in {len(pred)} predictions')

Encoding 10 validation queries...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Running validation...


Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

...

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


Validation Macro F1: 0.06669
Baseline:            0.06606

  Query 1: F1=0.1493 | P=0.2000 R=0.1190 | pred=25 gold=42 tp=5
  Query 2: F1=0.0000 | P=0.0000 R=0.0000 | pred=12 gold=36 tp=0
  Query 3: F1=0.0519 | P=0.0667 R=0.0426 | pred=30 gold=47 tp=2
  Query 4: F1=0.0645 | P=0.0476 R=0.1000 | pred=21 gold=10 tp=1
  Query 5: F1=0.0645 | P=0.0500 R=0.0909 | pred=20 gold=11 tp=1
  Query 6: F1=0.0000 | P=0.0000 R=0.0000 | pred=10 gold=18 tp=0
  Query 7: F1=0.0606 | P=0.0714 R=0.0526 | pred=14 gold=19 tp=1
  Query 8: F1=0.0488 | P=0.0833 R=0.0345 | pred=12 gold=29 tp=1
  Query 9: F1=0.2273 | P=0.1667 R=0.3571 | pred=30 gold=14 tp=5
  Query 10: F1=0.0000 | P=0.0000 R=0.0000 | pred=10 gold=25 tp=0

--- Retrieval Recall Debug ---
  Q1: found 5/42 gold citations in 25 predictions
  Q2: found 0/36 gold citations in 12 predictions
  Q3: found 2/47 gold citations in 30 predictions
  Q4: found 1/10 gold citations in 21 predictions
  Q5: found 1/11 gold citations in 20 predictions
  Q6: found 0/18 

## Generate Submission

In [14]:
start_time = time.time()

test_queries = test_df['query'].tolist()
print(f'Encoding {len(test_queries)} test queries...')
test_query_embeddings = encode_queries(embedding_model, test_queries)

print('Retrieving citations...')
submission_data = []
for i, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Test queries'):
    citations = retrieve_citations(row['query'], test_query_embeddings[i])
    submission_data.append({
        'query_id': row['query_id'],
        'predicted_citations': ';'.join(citations),
    })

submission_df = pd.DataFrame(submission_data)
submission_df.to_csv(OUTPUT_PATH, index=False)

elapsed = time.time() - start_time
print(f'\nCompleted in {elapsed/60:.1f} minutes')
print(f'Saved {len(submission_df)} predictions to {OUTPUT_PATH}')
submission_df.head()

Encoding 40 test queries...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Retrieving citations...


Test queries:   0%|          | 0/40 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

...

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]

Split strings:   0%|          | 0/1 [00:00<?, ?it/s]

BM25S Retrieve:   0%|          | 0/1 [00:00<?, ?it/s]


Completed in 3.5 minutes
Saved 40 predictions to submission.csv


,query_id,predicted_citations
0,test_001,Art. 5 Abs. 1 ZPO;Art. 22 Abs. 3 WSchG;Art. 12...
1,test_002,Art. 59 Abs. 1 SVG;Art. 83 Abs. 1 SVG;Art. 59 ...
2,test_003,Art. 8 Abs. 1 KKG;Art. 1 Abs. 2 KKG;Art. 11 Ab...
3,test_004,Art. 166 Abs. 2 SchKG;Art. 166 Abs. 1 SchKG;Ar...
4,test_005,Art. 1019 Abs. 2 OR;Art. 1178 Abs. 2 OR;Art. 7...
